In [ ]:
# 1. IMPORTS

import pandas as pd
import numpy as np

In [ ]:

# 2. LOAD DATA

train = pd.read_csv("train.csv")


# 3. BASIC CHECKS

print("Missing Values:\n", train.isnull().sum())
print("\nData Types:\n", train.dtypes)


# 4. KEEP ORIGINAL INCOME 


train['income_category'] = train['income'].str.strip()

# Fix inconsistent labels
train['income_category'] = train['income_category'].replace({
    '<=2L': '0-2L',
    '2L-5L': '2-5L',
    'More than 10L': '10L+'
})
=
# 5. FIX INCOME (ENCODING)

# Now create proper mapping
income_map = {
    '0-2L': 1,
    '2-5L': 2,
    '5L-10L': 3,
    '10L+': 4
}

train['income'] = train['income_category'].map(income_map)
train['income'] = train['income'].fillna(2) # Keep fillna for robustness

# User requested: print unique income mappings
print("\nIncome Category Mapping:\n", train[['income', 'income_category']].drop_duplicates())


# 6. FIX NUM_POLICIES 

def fix_policies(x):
    x = str(x)
    if x.isdigit():
        return int(x)
    elif 'More than 3' in x:
        return 4
    elif 'More than 2' in x:
        return 3
    else:
        return 2

train['num_policies'] = train['num_policies'].apply(fix_policies)


# 7. SCALE INCOME (CRITICAL FIX)

train['income_scaled'] = train['income'] * 500000


# 8. FEATURE ENGINEERING 


# Claim ratio 
train['claim_ratio'] = train['claim_amount'] / (train['income_scaled'] + 1)


train['claim_percentage'] = train['claim_ratio'] * 100
# round claim_percentage
train['claim_percentage'] = train['claim_percentage'].round(2)

# Engagement
train['policies_per_year'] = train['num_policies'] / (train['vintage'] + 1)

# Customer value
train['customer_value'] = (
    train['income_scaled'] *
    train['num_policies'] *
    (train['vintage'] + 1)
)

# High value flag
train['high_value_flag'] = ((train['income'] >= 3) & (train['num_policies'] > 1)).astype(int)

# 9. CUSTOMER SEGMENTATION 

train['customer_segment'] = pd.qcut(
    train['cltv'],
    3,
    labels=['Low', 'Medium', 'High']
)

# 10. HANDLE MISSING VALUES

train['claim_amount'] = train['claim_amount'].fillna(0)
train['vintage'] = train['vintage'].fillna(train['vintage'].median())


# 11. FINAL VALIDATION 

print("\nSample Data:\n", train.head())

print("\nCheck Claim Ratio Range:")
print(train['claim_ratio'].describe())

print("\nCheck Customer Value:")
print(train['customer_value'].describe())

print("\nSegment Distribution:")
print(train['customer_segment'].value_counts())


# 12. EXPORT CLEAN DATA

train.to_csv("cleaned_train.csv", index=False)


In [ ]:

cleaned_train_df = pd.read_csv('cleaned_train.csv')
display(cleaned_train_df.head())

,id,gender,area,qualification,income,marital_status,vintage,claim_amount,num_policies,policy,type_of_policy,cltv,income_category,income_scaled,claim_ratio,claim_percentage,policies_per_year,customer_value,high_value_flag,customer_segment
0,1,Male,Urban,Bachelor,2.0,1,5,5790,2,A,Platinum,64308,5L-10L,1000000.0,0.005790,0.578999,0.333333,2000000.0,0,Medium
1,2,Male,Rural,High School,2.0,0,8,5080,2,A,Platinum,515400,5L-10L,1000000.0,0.005080,0.507999,0.222222,2000000.0,0,High
2,3,Male,Urban,Bachelor,2.0,1,8,2599,2,A,Platinum,64212,5L-10L,1000000.0,0.002599,0.259900,0.222222,2000000.0,0,Medium
3,4,Female,Rural,High School,2.0,0,7,0,2,A,Platinum,97920,5L-10L,1000000.0,0.000000,0.000000,0.250000,2000000.0,0,High
4,5,Male,Urban,High School,2.0,1,6,3508,2,A,Gold,59736,More than 10L,1000000.0,0.003508,0.350800,0.285714,2000000.0,0,Low


In [34]:
print("Summary Statistics for Numerical Columns:")
print(train.describe())

Summary Statistics for Numerical Columns:
                 id        income  marital_status       vintage  claim_amount  \
count  89392.000000  89392.000000    89392.000000  89392.000000  89392.000000   
mean   44696.500000      2.874687        0.575488      4.595669   4351.502416   
std    25805.391969      0.675873        0.494272      2.290446   3262.359775   
min        1.000000      1.000000        0.000000      0.000000      0.000000   
25%    22348.750000      2.000000        0.000000      3.000000   2406.000000   
50%    44696.500000      3.000000        1.000000      5.000000   4089.000000   
75%    67044.250000      3.000000        1.000000      6.000000   6094.000000   
max    89392.000000      4.000000        1.000000      8.000000  31894.000000   

       num_policies           cltv  income_scaled   claim_ratio  \
count  89392.000000   89392.000000   8.939200e+04  89392.000000   
mean       1.674143   97952.828978   1.437343e+06      0.003396   
std        0.468697   90613